# unbox-args-tensor-to-array — worked example 2: Unbox MiniTensor kwargs

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `unbox-args-tensor-to-array`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The same unbox rule applies to keyword arguments: each `MiniTensor`-valued entry is replaced by its `.array`, and every other value passes through. Keys and insertion order are preserved, and a brand-new dict is returned so the caller's kwargs are never mutated.

## Worked solution

We unbox a kwargs dict for the raw forward call.

1. We build a new dict via comprehension over `kwargs.items()`, preserving keys and Python 3.7+ insertion order.
2. For each value, `isinstance(v, MiniTensor)` decides whether to substitute `v.array`; non-Tensor values like an int `dim` stay unchanged.
3. We use the `isinstance` gate (not `.array` duck-typing) so a raw array value would pass through untouched.
4. The original `kwargs` dict is left intact because we never write into it.

We print the unboxed dict and confirm the tensor value became its `.array`.

In [ ]:
class MiniTensor:
    def __init__(self, array):
        self.array = array

def unbox_kwargs(kwargs: dict) -> dict:
    return {k: (v.array if isinstance(v, MiniTensor) else v) for k, v in kwargs.items()}

src = MiniTensor([7, 8, 9])
out = unbox_kwargs({'src': src, 'dim': 1})
print('unboxed:', out)
print('src unboxed:', out['src'] is src.array)
print('dim passthrough:', out['dim'] == 1)